In [18]:
%pip install "qiskit==1.4.3" "qiskit-ibm-runtime==0.40.1" "qiskit-nature==0.7.2" "qiskit-algorithms==0.3.1" "pyscf==2.8.0"

In [19]:
import os, sys, json, time, math, traceback, shutil, datetime as dt, importlib.metadata as md
import numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2

SECRET_NAME = "IBM_QUANTUM_API_KEY"
ALLOWED_BACKENDS = ["ibm_kingston", "ibm_fez", "ibm_marrakesh"]
BACKEND_NAME = "auto"
INSTANCE_CRN = ""
PREC_TARGET = 0.005      # target standard error in Ha
RESILIENCE_LEVEL = 1     # TREX readout mitigation + twirling (cheap)
MAX_QPU_SECONDS = 240
BUDGET_S = 100           # QPU time I want to plan for
WAIT_TIMEOUT_S = 1800
R1, R2, ANGLE = 0.96, 2.20, 104.5
SAVED_CASCI = -74.860977098519

KEY = None; LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)
def safe(e):
    s = f"{type(e).__name__}: {e}"
    return s.replace(KEY, "***") if KEY else s

for p in ["qiskit", "qiskit-ibm-runtime", "qiskit-nature", "pyscf", "numpy"]:
    log(f"{p}=={md.version(p)}")

from google.colab import userdata
KEY = userdata.get(SECRET_NAME)
if not KEY: raise RuntimeError("Secret IBM_QUANTUM_API_KEY missing or Notebook access off.")
try: INSTANCE_CRN = INSTANCE_CRN or (userdata.get("IBM_QUANTUM_INSTANCE") or "")
except Exception: pass

service = None
for ch in ("ibm_quantum_platform", "ibm_cloud"):
    try:
        service = QiskitRuntimeService(channel=ch, token=KEY, instance=(INSTANCE_CRN or None)); log("Connected:", ch); break
    except Exception as e:
        if "channel" in str(e).lower(): continue
        raise RuntimeError("IBM connection failed: " + safe(e)) from None
avail = {b.name: b for b in service.backends(simulator=False, operational=True)}
cands = [n for n in ALLOWED_BACKENDS if n in avail]
if not cands: raise RuntimeError("No hackathon backend available.")
pend = {n: avail[n].status().pending_jobs for n in cands}; log("Pending:", pend)
sel = min(pend, key=pend.get) if BACKEND_NAME == "auto" else BACKEND_NAME
backend = avail[sel]
try: CAL_TIME = str(backend.properties().last_update_date)
except Exception: CAL_TIME = None
log(f"BACKEND {backend.name} | {backend.num_qubits} qubits | calibration {CAL_TIME}")

qiskit==1.4.3
qiskit-ibm-runtime==0.40.1
qiskit-nature==0.7.2
pyscf==2.8.0
numpy==2.1.3


qiskit_runtime_service._resolve_cloud_instances:WARNING:2026-10-24 21:22:44,495: Default instance not set. Searching all available instances.


Connected: ibm_quantum_platform
Pending: {'ibm_kingston': 1, 'ibm_fez': 2, 'ibm_marrakesh': 798}
BACKEND ibm_kingston | 156 qubits | calibration 2026-10-24 20:25:26+00:00


In [20]:
from pyscf import gto, scf, mcscf
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock
from qiskit.quantum_info import Statevector, SparsePauliOp

hx, hz = R2*np.sin(np.radians(ANGLE)), R2*np.cos(np.radians(ANGLE))
ATOM = f"O 0 0 0; H 0 0 {R1}; H {hx:.12f} 0 {hz:.12f}"
mol = gto.M(atom=ATOM, basis="sto3g", unit="Angstrom", verbose=0)
mf = scf.RHF(mol); mf.conv_tol = 1e-10; mf.kernel()
E_RHF = float(mf.e_tot)
E_CASCI = float(mcscf.CASCI(mf, 6, 8).kernel()[0])
E_FCI_FULL = float(mcscf.CASCI(mf, 7, 10).kernel()[0])

driver = PySCFDriver(atom=ATOM, basis="sto3g", charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
problem = ActiveSpaceTransformer(num_electrons=8, num_spatial_orbitals=6).transform(driver.run())
consts = {k: float(v) for k, v in problem.hamiltonian.constants.items()}
CONST = float(sum(consts.values()))
mapper = ParityMapper(num_particles=problem.num_particles)
qc_ = mapper.map(problem.hamiltonian.second_q_op())
assert np.max(np.abs(qc_.coeffs.imag)) < 1e-10
qop = SparsePauliOp(qc_.paulis, qc_.coeffs.real).simplify()
_, aux_ops = problem.second_q_ops()
aux_q = {k: mapper.map(v) for k, v in aux_ops.items() if k in ("ParticleNumber", "AngularMomentum")}

w, V = np.linalg.eigh(qop.to_matrix())
Nm, S2m = aux_q["ParticleNumber"].to_matrix(), aux_q["AngularMomentum"].to_matrix()
E_EXACT = None
for i in range(len(w)):
    v = V[:, i]
    if abs((v.conj()@Nm@v).real - 8) < 1e-6 and abs((v.conj()@S2m@v).real) < 1e-6:
        E_EXACT = float(w[i] + CONST); break

hf = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
sv = Statevector(hf)
E_HF_IDEAL = float(sv.expectation_value(qop).real + CONST)
N_HF = float(sv.expectation_value(aux_q["ParticleNumber"]).real)
pmax = max(sv.probabilities_dict().values())

fails = []
def check(n, ok, d):
    log(("PASS " if ok else "FAIL ") + n + " | " + d)
    if not ok: fails.append(n)
check("10 qubits", qop.num_qubits == 10, str(qop.num_qubits))
check("N=8", abs(N_HF - 8) < 1e-9, f"{N_HF:.6f}")
check("HF circuit == RHF", abs(E_HF_IDEAL - E_RHF) < 1e-6, f"{E_HF_IDEAL:.10f} vs {E_RHF:.10f}")
check("mapped exact == CASCI", E_EXACT is not None and abs(E_EXACT - E_CASCI) < 1e-6, f"{E_EXACT} vs {E_CASCI:.12f}")
check("CASCI == saved", abs(E_CASCI - SAVED_CASCI) < 1e-6, f"{E_CASCI:.12f}")
check("HF state is a single basis state", pmax > 1 - 1e-9, f"max prob {pmax:.12f}")
log(f"Constants (added once): {consts} total {CONST:.9f}")
log(f"RHF {E_RHF:.9f} | CASCI {E_CASCI:.9f} | FCI(full STO-3G, different ref) {E_FCI_FULL:.9f}")
if fails: raise RuntimeError("Checks failed: " + ", ".join(fails))

PASS 10 qubits | 10
PASS N=8 | 8.000000
PASS HF circuit == RHF | -74.6238923807 vs -74.6238923807
PASS mapped exact == CASCI | -74.8609770985192 vs -74.860977098519
PASS CASCI == saved | -74.860977098519
PASS HF state is a single basis state | max prob 1.000000000000
Constants (added once): {'nuclear_repulsion_energy': 6.536735094058561, 'ActiveSpaceTransformer': -60.03401939624077} total -53.497284302
RHF -74.623892381 | CASCI -74.860977099 | FCI(full STO-3G, different ref) -74.861014224


In [21]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

keep = np.flatnonzero(~qop.paulis.x.any(axis=1)).tolist()   # terms with only I/Z
qop_diag = qop[keep].simplify()
E_DIAG_IDEAL = float(sv.expectation_value(qop_diag).real + CONST)
groups = qop_diag.group_commuting(qubit_wise=True)
check("Z-only part reproduces full HF energy exactly", abs(E_DIAG_IDEAL - E_HF_IDEAL) < 1e-9, f"{E_DIAG_IDEAL:.10f}")
check("Z-only part is ONE measurement group", len(groups) == 1, f"{len(groups)} group(s), {len(qop_diag)} of {len(qop)} terms")
if fails: raise RuntimeError("Checks failed: " + ", ".join(fails))

pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=7)
isa = pm.run(hf)
isa_obs = qop_diag.apply_layout(isa.layout)
ops = dict(isa.count_ops())
bad = sorted(set(ops) - set(backend.operation_names) - {"barrier"})
assert isa.num_nonlocal_gates() == 0 and not bad, f"unexpected ops {ops} {bad}"

nz = [not (p.x.any() or p.z.any()) for p in qop_diag.paulis]
wq = float(np.sqrt(np.sum(np.abs(qop_diag.coeffs[~np.array(nz)])**2)))
T_SHOT = 300e-6
max_shots = BUDGET_S / T_SHOT
PRECISION = max(PREC_TARGET, wq / math.sqrt(max_shots))
est_shots = (wq / PRECISION)**2; est_s = est_shots * T_SHOT
OPTS = {"default_precision": PRECISION, "resilience_level": RESILIENCE_LEVEL, "max_execution_time": int(MAX_QPU_SECONDS)}
EstimatorV2(mode=backend, options=OPTS)    # validates options against installed version
log(f"Circuit ops {ops} | physical qubits {len(isa.layout.final_index_layout())} of {backend.num_qubits}")
log(f"Precision used {PRECISION:.4f} Ha (target {PREC_TARGET}) | rough est {est_shots:,.0f} shots, ~{est_s:.0f} s QPU")
log(f"Ideal HF energy to compare with: {E_HF_IDEAL:.6f} Ha | PREFLIGHT DONE, NOTHING SENT TO HARDWARE")

PASS Z-only part reproduces full HF energy exactly | -74.6238923807
PASS Z-only part is ONE measurement group | 1 group(s), 79 of 1079 terms
Circuit ops {'x': 4} | physical qubits 10 of 156
Precision used 0.0050 Ha (target 0.005) | rough est 313,436 shots, ~94 s QPU
Ideal HF energy to compare with: -74.623892 Ha | PREFLIGHT DONE, NOTHING SENT TO HARDWARE


In [22]:
GO = False       # <<< set True to send ONE real job

RESULT = {"kind": "DRY_RUN_NO_HARDWARE", "job_id": None, "status": "NOT_SUBMITTED"}
if not GO:
    log("GO is False -> not submitted.")
else:
    try:
        estimator = EstimatorV2(mode=backend, options=OPTS)
        log(f"Submitting ONE job on {backend.name} (HF state, Z-only observable, precision {PRECISION:.4f}) ...")
        t0 = time.time()
        job = estimator.run([(isa, isa_obs)], precision=PRECISION)
        RESULT.update(job_id=job.job_id(), status="SUBMITTED", kind="REAL_HARDWARE_HF_ZONLY")
        log("JOB ID:", job.job_id())
        last = None
        while not job.in_final_state():
            st = str(getattr(job.status(), "name", job.status()))
            if st != last: log(f"  [{time.time()-t0:5.0f}s] {st}"); last = st
            if time.time() - t0 > WAIT_TIMEOUT_S:
                RESULT["status"] = "CLIENT_TIMEOUT"; log("Timeout; job not cancelled:", job.job_id()); break
            time.sleep(5)
        if job.in_final_state():
            st = str(getattr(job.status(), "name", job.status())); RESULT["status"] = st
            log(f"Final status {st} after {time.time()-t0:.0f}s")
            if st == "DONE":
                pub = job.result()[0]
                ev = float(np.asarray(pub.data.evs).ravel()[0]); sd = float(np.asarray(pub.data.stds).ravel()[0])
                RESULT.update(ev=ev, std=sd, total_energy=ev + CONST, metadata=json.loads(json.dumps(pub.metadata, default=str)))
                try: RESULT["usage"] = json.loads(json.dumps(job.usage(), default=str))
                except Exception: pass
                log(f"MEASURED on {backend.name}: E_total = {ev+CONST:.6f} Ha, std error {sd*1e3:.2f} mHa (statistical only)")
                log(f"Ideal HF {E_HF_IDEAL:.6f} Ha | measured - ideal HF = {(ev+CONST-E_HF_IDEAL)*1e3:+.2f} mHa | measured - CASCI = {(ev+CONST-E_CASCI)*1e3:+.2f} mHa")
            else:
                try: log("Job error:", job.error_message())
                except Exception: pass
    except Exception as e:
        log("RUN ERROR (no resubmission):", safe(e)); log(traceback.format_exc().replace(KEY, "***"))

GO is False -> not submitted.


In [23]:
GO = True       # sends ONE real job

RESULT = {"kind": "DRY_RUN_NO_HARDWARE", "job_id": None, "status": "NOT_SUBMITTED"}
if not GO:
    log("GO is False -> not submitted.")
else:
    try:
        estimator = EstimatorV2(mode=backend, options=OPTS)
        log(f"Submitting ONE job on {backend.name} (HF state, Z-only observable, precision {PRECISION:.4f}) ...")
        t0 = time.time()
        job = estimator.run([(isa, isa_obs)], precision=PRECISION)
        RESULT.update(job_id=job.job_id(), status="SUBMITTED", kind="REAL_HARDWARE_HF_ZONLY")
        log("JOB ID:", job.job_id())
        last = None
        while not job.in_final_state():
            st = str(getattr(job.status(), "name", job.status()))
            if st != last: log(f"  [{time.time()-t0:5.0f}s] {st}"); last = st
            if time.time() - t0 > WAIT_TIMEOUT_S:
                RESULT["status"] = "CLIENT_TIMEOUT"; log("Timeout; job not cancelled:", job.job_id()); break
            time.sleep(5)
        if job.in_final_state():
            st = str(getattr(job.status(), "name", job.status())); RESULT["status"] = st
            log(f"Final status {st} after {time.time()-t0:.0f}s")
            if st == "DONE":
                pub = job.result()[0]
                ev = float(np.asarray(pub.data.evs).ravel()[0]); sd = float(np.asarray(pub.data.stds).ravel()[0])
                RESULT.update(ev=ev, std=sd, total_energy=ev + CONST, metadata=json.loads(json.dumps(pub.metadata, default=str)))
                try: RESULT["usage"] = json.loads(json.dumps(job.usage(), default=str))
                except Exception: pass
                log(f"MEASURED on {backend.name}: E_total = {ev+CONST:.6f} Ha, std error {sd*1e3:.2f} mHa (statistical only)")
                log(f"Ideal HF {E_HF_IDEAL:.6f} Ha | measured - ideal HF = {(ev+CONST-E_HF_IDEAL)*1e3:+.2f} mHa | measured - CASCI = {(ev+CONST-E_CASCI)*1e3:+.2f} mHa")
            else:
                try: log("Job error:", job.error_message())
                except Exception: pass
    except Exception as e:
        log("RUN ERROR (no resubmission):", safe(e)); log(traceback.format_exc().replace(KEY, "***"))

Submitting ONE job on ibm_kingston (HF state, Z-only observable, precision 0.0050) ...
JOB ID: dbei45vpc3tc739lomf0
  [    4s] RUNNING
Final status DONE after 70s
MEASURED on ibm_kingston: E_total = -74.542299 Ha, std error 2140.26 mHa (statistical only)
Ideal HF -74.623892 Ha | measured - ideal HF = +81.59 mHa | measured - CASCI = +318.68 mHa


In [24]:
from qiskit import qpy
ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = f"/content/h2o_hw_{ts}" if os.path.isdir("/content") else f"h2o_hw_{ts}"
os.makedirs(OUT, exist_ok=True)
label = ("REAL HARDWARE JOB: HF reference state, Z-only (diagonal) observable. Not a VQE result." if RESULT["kind"].startswith("REAL")
         else "DRY RUN: no hardware measurement exists.")
summary = {"label": label, "geometry": ATOM, "basis": "STO-3G", "active_space": "CAS(8e,6o), O 1s frozen",
           "mapper": "ParityMapper(num_particles), 10 qubits",
           "refs_Ha": {"RHF": E_RHF, "CASCI": E_CASCI, "FCI_full_STO3G": E_FCI_FULL, "HF_ideal": E_HF_IDEAL},
           "constants_Ha": consts, "backend": backend.name, "calibration": CAL_TIME,
           "options": OPTS, "ops": ops, "result": RESULT,
           "packages": {p: md.version(p) for p in ["qiskit", "qiskit-ibm-runtime", "qiskit-nature", "pyscf"]},
           "limitations": ["Only Z-only terms measured; X/Y terms are exactly 0 for the ideal HF basis state but not verified on hardware.",
                           "Std error is statistical only; readout and gate bias beyond TREX are not included.",
                           "HF state is 237 mHa above CASCI by construction."]}
blob = json.dumps(summary, indent=2, default=str); logtxt = "\n".join(LOG)
assert KEY not in blob and KEY not in logtxt
open(f"{OUT}/summary.json", "w").write(blob); open(f"{OUT}/console.log", "w").write(logtxt)
with open(f"{OUT}/transpiled.qpy", "wb") as f: qpy.dump([isa], f)
zp = shutil.make_archive(OUT, "zip", OUT); print("Saved", zp, "\n", label)
try:
    from google.colab import files; files.download(zp)
except Exception: pass

Saved /content/h2o_hw_20261024_212656.zip 
 REAL HARDWARE JOB: HF reference state, Z-only (diagonal) observable. Not a VQE result.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
print(RESULT.get("usage")); print(json.dumps(RESULT.get("metadata"), indent=1)[:2500])

21
{
 "shots": 40000,
 "target_precision": 0.005,
 "circuit_metadata": {},
 "resilience": {},
 "num_randomizations": 32
}


In [26]:
OPTS2 = {"resilience_level": 0, "default_shots": 100000, "max_execution_time": 120}
estimator2 = EstimatorV2(mode=backend, options=OPTS2)
job2 = estimator2.run([(isa, isa_obs)])
log("JOB2 ID:", job2.job_id())
while not job2.in_final_state(): time.sleep(5)
p2 = job2.result()[0]
ev2 = float(np.asarray(p2.data.evs).ravel()[0]); sd2 = float(np.asarray(p2.data.stds).ravel()[0])
log(f"RAW (no mitigation) E_total = {ev2+CONST:.6f} Ha, std {sd2*1e3:.2f} mHa, vs ideal HF {(ev2+CONST-E_HF_IDEAL)*1e3:+.2f} mHa")
RESULT["job2"] = {"id": job2.job_id(), "E_total": ev2 + CONST, "std": sd2, "options": OPTS2}

JOB2 ID: dbei5tvpc3tc739lop2g
RAW (no mitigation) E_total = -74.327453 Ha, std 3.45 mHa, vs ideal HF +296.44 mHa


In [27]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
ro = {}
for q in range(backend.num_qubits):
    try: ro[q] = backend.target["measure"][(q,)].error
    except Exception: pass
best = sorted(ro, key=ro.get)[:10]
log("Best-readout qubits:", best, "| mean readout error", np.mean([ro[q] for q in best]), "vs device mean", np.mean(list(ro.values())))

pm_b = generate_preset_pass_manager(backend=backend, optimization_level=1, initial_layout=best, seed_transpiler=7)
isa_b = pm_b.run(hf)
obs_b = qop_diag.apply_layout(isa_b.layout)
assert isa_b.num_nonlocal_gates() == 0

OPTS3 = {"resilience_level": 0, "default_shots": 100000, "max_execution_time": 120}
job3 = EstimatorV2(mode=backend, options=OPTS3).run([(isa_b, obs_b)])
log("JOB3 ID:", job3.job_id())
while not job3.in_final_state(): time.sleep(5)
p3 = job3.result()[0]
ev3 = float(np.asarray(p3.data.evs).ravel()[0]); sd3 = float(np.asarray(p3.data.stds).ravel()[0])
log(f"RAW, best qubits: E_total = {ev3+CONST:.6f} Ha, std {sd3*1e3:.2f} mHa, vs ideal HF {(ev3+CONST-E_HF_IDEAL)*1e3:+.2f} mHa")
RESULT["job3"] = {"id": job3.job_id(), "qubits": best, "E_total": ev3 + CONST, "std": sd3, "options": OPTS3}

Best-readout qubits: [148, 42, 117, 149, 154, 44, 91, 134, 40, 89] | mean readout error 0.0031005859375 vs device mean 0.020158816606570512
JOB3 ID: dbei8tom94ac738rsp90
RAW, best qubits: E_total = -74.573420 Ha, std 1.47 mHa, vs ideal HF +50.47 mHa
